## Introduction to the extended version of DiCE (Diverse Counterfactual Explanations)

[Mothilal et al. (2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) introduce their method of generating counterfactual explanations considering _feasibility_, and _diversity_. [Guidotti and Ruggieri (2021)](https://link.springer.com/chapter/10.1007/978-3-030-88942-5_28), claim counterfactual explanations to be robust they should be similar for similar instances when they explain. In this study, in a search to improve the quality and reliability of the counterfactual explanations _robustness_ is found to be helpful and it also introduced in the optimization function.

DiCE-Extended is built upon the [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) [(Mothilal et al. 2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) framework by introducing a robustness term in the optimization function.

## Manipulated Optimization Function

The core enhancement in DiCE-Extended is the manipulated optimization function, designed to balance proximity, diversity, and feasibility of counterfactuals. The function is formulated as:

<a id="equation-1"></a>
\begin{equation}
\tag{1}
C(x) = \underset{c_1, ..., c_k}{\text{arg min}}
\frac{1}{2} \sum_{i=1}^{k} yloss(f(c_i), y) +
\frac{\lambda_1}{k} \sum_{i=1}^{k} dist(c_i, x) -
\lambda_2 \cdot dpp\_diversity(c_1, ..., c_k) -
\frac{\lambda_3}{k} \sum_{i=1}^{k} robustness(c_i, c_i')
\end{equation}

- **Proximity Loss**: The first term that averages the distance between generated counterfactuals and the original input ensure the counterfactuals to be as close as possible to the original input.
- **Diversity Loss**: Diversity of the counterfactual explanations is aquired by determinental point process of which loss is represented by the second term and it ensures that _k_ number of counterfactual explanations are generated.
- **Robustness Loss**: [Guidotti (2024)](https://link.springer.com/article/10.1007/s10618-022-00831-6) defines robustness as necessity of similar instances being explained by similar counterfactual explanations such that if $b(x_1)=b(x_2)=y$ then an explainer $f$ should generate counterfactuals $c_1$ and $c_2$ that are similar and can explain $x_1$ and $x_2$. The robustness term that is based on [Dice-Sørensen Coefficient](https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient), is adopted from [Bonasera and Carrizosa (2024)](
https://doi.org/10.48550/arXiv.2407.00843).

\begin{equation}
\tag{2}
Robustness(c_i, c_i') = \frac{2 * \lvert c_i \cap c_i' \rvert}{\lvert c_i \rvert + \lvert c_i' \rvert}
\end{equation}


By adjusting the weights $\lambda_1$, $\lambda_2$, $\lambda_3$ counterfactual explanations can be customised by specific needs.

## Metrics and Sensitivity Analysis for Dice Extended


### 1. Robustness Metrics

#### Dice-Sørensen Coefficient

To evaluate robustness, the Dice-Sørensen coefficient measures the similarity between counterfactuals c1 and
c2 generated for similar input instances x1 and x2:

\begin{equation}
\tag{3}
Robustness(c_1, c_2) = \frac{2 * \lvert c_1 \cap c_2 \rvert}{\lvert c_1 \rvert + \lvert c_2 \rvert}
\end{equation}

where:
- $ c_1 $ and $ c_2 $ are binary vectors,
- $ \lvert c_1 \cap c_2 \rvert $: The number of shared (overlapping) features between c1 and c2,
- $ \lvert c_1 \rvert $ and $ \lvert c_2 \rvert $: The total number of features in each counterfactual.

#### Input Perturbation and Stability

Stability under input perturbation measures the solution variance when slight perturbations are introduced
to the input instance. The procedure includes the following steps:

1) **Apply Gaussian Noise:** Perturb the input $x$ by adding Gaussian noise $\delta$ to create perturbed inputs
$x'$:

\begin{equation}
\tag{4}
x' = x + \delta, \quad \delta \sim \mathcal{N}(0, \sigma^2)
\end{equation}

where $\sigma$ is the standard deviation of the noise (e.g., $\sigma = 0.01$).

2) **Generate Counterfactuals:** Generate counterfactual explanations $c_i$ for the original input $x$ and $c_i'$ for the perturbed input $x'$.

3) **Measure Stability:** Compare counterfactuals using a distance metric, such as the Euclidean distance:

\begin{equation}
\tag{5}
Stability = \frac{1}{n} \sum_{i=1}^{n} dist(c_i, c_i')
\end{equation}

where:

\begin{equation}
\tag{6}
dist(c_i, c_i') = \sqrt{\sum_{j=1}^{d} (c_{ij} - c_{ij}')^2}
\end{equation}

$n$ is the total number of input instances, $c_i$ is the counterfactual for the original input, and $c_i'$ is the counterfactual for the perturbed input.

### 2. Counterfactual Quality Measures

#### Fidelity

Fidelity measures how often generated counterfactuals successfully change the model’s prediction:

\begin{equation}
\tag{7}
Fidelity = \frac{\sum_{i=1}^{n} \mathbf{1}(f(c_i) = y_{desired})}{n}
\end{equation}

where:

- $f$: Prediction model,
- $c_i$: Counterfactual instance,
- $y_{desired}$: Target output class,
- $n$: Total number of counterfactuals.

#### Proximity

Proximity measures the average distance between counterfactuals $c_i$ and the original inputs $x_i$:

\begin{equation}
\tag{8}
Proximity = \frac{1}{n} \sum_{i=1}^{n} dist(x_i, c_i)
\end{equation}

The Manhattan distance can be used for simplicity:

\begin{equation}
\tag{9}
dist(x_i, c_i) = \sum_{j=1}^{d} \lvert x_{ij} - c_{ij} \rvert
\end{equation}

#### Diversity

Diversity measures how dissimilar the counterfactuals $c_1, c_2, c_3,\ldots,c_k$ are among themselves:

\begin{equation}
\tag{10}
Diversity = \frac{1}{k(k-1)}\sum_{i_1}^{k}\sum_{j \neq i}^{} dist(c_i, c_j)
\end{equation}

where $k$ is the number of counterfactuals.

### 3. Sensitivity Analysis

#### Objective Function with Weights

The modified loss function in DiCE-Extended is defined as in the [equation 1](#equation-1) where:

- $yloss(f(c_i), y)$: Prediction loss for counterfactual instance $c_i$ relative to the desired outcome $y$,
- $dist(c_i, x)$: Distance metric (e.g., Euclidean or Manhattan) between the counterfactual c_i and the original input $x$,
- $dpp_diversity(c_1,\ldots,c_k)$: Diversity loss term based on Determinantal Point Process (DPP),
- $Robustness(c_i,c_i')$: Robustness loss measuring similarity of counterfactuals under perturbations.

  The weights $\lambda_1, \lambda_2, \lambda_3$ control the balance between proximity, diversity, and robustness, respectively.

#### Sensitivity Analysis

To perform sensitivity analysis:

1) Vary the weights $\lambda_1, \lambda_2, \lambda_3$ systematically while ensuring:

\begin{equation}
\tag{11}
\lambda_1 + \lambda_2 + \lambda_3 = 1 (for normalization).
\end{equation}

2) Track the changes in the following metrics:

\begin{equation}
\tag{12}
P(\lambda_1, \lambda_2, \lambda_3) = Proximity,
\end{equation}

\begin{equation}
\tag{13}
D(\lambda_1, \lambda_2, \lambda_3) = Diversity,
\end{equation}

\begin{equation}
\tag{14}
R(\lambda_1, \lambda_2, \lambda_3) = Robustness,
\end{equation}

3) Measure the relationship between these metrics and the weights.



In [1]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [2]:
import numpy as np
import timeit
import random
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from torch.utils.data import DataLoader

import dice_ml_x
from dice_ml_x.utils import helpers
from dice_ml_x.utils import neuralnetworks
from dice_ml_x import DiceX

In [3]:
%load_ext autoreload
%autoreload 2

In [27]:
datasets = ["compas", "adult-income", "german-credit-risk", "lending-club"]
from dice_ml_x.utils import helpers
    

df_lending = helpers.load_lending_club_dataset()
cat_columns = df_lending.columns.difference(list(df_lending.select_dtypes(include=[np.number])))
df_lending['stratify_column'] = df_lending[cat_columns].astype(str).agg('_'.join, axis=1)



# Filter the target variable to match the filtered dataframe
target = df_lending["loan_status"]

train_dataset, test_dataset, y_train, y_test = train_test_split(df_lending,
                                                                target,
                                                                test_size=0.2,
                                                                random_state=42,
                                                                stratify=target)
x_train = train_dataset.drop(columns=['stratify_column'])
x_test = test_dataset.drop(columns=['stratify_column'])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, cat_columns)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', RandomForestClassifier())])
model = clf.fit(x_train, y_train)


In [39]:
df_lending = helpers.load_lending_club_dataset()
numerical = df_lending.select_dtypes(include=[np.number]).columns

categorical = df_lending.columns.difference(numerical).difference(['loan_status'])
import torch

from torch.utils.data import random_split, DataLoader
target = df_lending['loan_status']
features = df_lending.drop(columns=['loan_status'])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations)])

transformed_data = clf.fit_transform(features)
# Example dataset
dataset = torch.utils.data.TensorDataset(torch.tensor(transformed_data.toarray(), dtype=torch.float32),
                                         torch.tensor(df_lending['loan_status'].values))

# Split sizes
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Split
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [44]:
dummy_input, dummy_label = next(iter(train_loader))
in_features = dummy_input.shape[1]
pyt_model = neuralnetworks.PYTModel(in_features)
trainer = pyt_model.train(train_dataloader=train_loader, test_dataloader=test_loader)

In [ ]:
d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=["age", "juv_misd_count", "priors_count"], outcome_name='twoyearrecid')

In [15]:
import pickle

with open("benchmarking_results.pkl", "wb") as f:
    pickle.dump(benchmarking.results, f)

In [11]:
benchmarking.results
m = dice_ml_x.Model(model=benchmarking.results['compas-recidivism']['PYT'], backend='PYT', func="ohe-min-max")

In [14]:
dummy_compas = helpers.load_compas_dataset()
c_target = dummy_compas['twoyearrecid']
c_train_dataset, c_test_dataset, c_y_train, c_y_test = train_test_split(dummy_compas, c_target, test_size=0.2,
                                                                        random_state=0,
                                                                        stratify=c_target)
compas_train_dataloader
m.model.model(c_test_dataset)

TypeError: linear(): argument 'input' (position 1) must be Tensor, not DataFrame

In [94]:

lending_df = helpers.load_lending_club_dataset()
target = lending_df['loan_status']

categorical_columns = lending_df.columns.difference(list(lending_df.select_dtypes(include=[np.number])))
lending_df['composite_stratify'] = lending_df[categorical_columns].astype(str).agg('_'.join, axis=1)

# Ensure each class in composite_stratify has at least two members
value_counts = lending_df['composite_stratify'].value_counts()
valid_classes = value_counts[value_counts >= 2].index
lending_df = lending_df[lending_df['composite_stratify'].isin(valid_classes)]

# Filter the target variable to match the filtered dataframe
target = target[lending_df.index]

# Split the dataset, ensuring all categories are present in both splits
train_dataset, test_dataset, y_train, y_test = train_test_split(lending_df, target, test_size=0.2, random_state=42, stratify=lending_df['composite_stratify'])

# Drop the composite column after splitting
train_df = train_dataset.drop(columns=['composite_stratify'])
test_df = test_dataset.drop(columns=['composite_stratify'])

train_sub_unique = train_dataset['sub_grade'].unique().tolist()
test_sub_unique = test_dataset['sub_grade'].unique().tolist()
print(set(train_sub_unique) - set(test_sub_unique))

ValueError: The test_size = 305 should be greater or equal to the number of classes = 682

In [63]:
import pickle
with open("benchmarking_results_fin.pkl", "wb") as f:
    pickle.dump(benchmarking.results, f)


In [12]:
benchmarking.results

{'compas-recidivism': {'sklearn': {'accuracy': 0.6197183098591549,
   'cfs': {'gaussian':     sex   age  juv_misd_count  priors_count        age_cat       race  \
    0  Male  49.0             0.0          10.0  Greaterthan45  Caucasian   
    0  Male  49.0             0.0           6.0  Greaterthan45  Caucasian   
    0  Male  53.0             0.0          10.0  Greaterthan45  Caucasian   
    0  Male  45.0             0.0          13.0  Greaterthan45  Caucasian   
    
      c_charge_degree  twoyearrecid  
    0               M             0  
    0               F             0  
    0               F             0  
    0               F             0  ,
    'random':     sex   age  juv_misd_count  priors_count        age_cat              race  \
    0  Male  47.2             0.0          10.0  Greaterthan45         Caucasian   
    0  Male  46.0             0.0          10.0  Greaterthan45         Caucasian   
    0  Male  19.0             0.0           0.0  Greaterthan45         

In [13]:
from dice_ml_x.benchmarking import Benchmarking
datasets = ["compas-recidivism", "adult-income", "lending-club", "german-credit"]
datasets = ["compas-recidivism", "adult-income"]
backends = ["sklearn", "PYT", "TF2"]
methods = ['gaussian', 'random', 'spherical']
benchmarking = Benchmarking(datasets=datasets,
                            backends=backends,
                            perturbation_methods=methods)
benchmarking.load_and_train(batch_size=16)
from datetime import datetime
import pickle
current_time = datetime.now()
formatted_timestamp = current_time.strftime("%d_%m_%Y-%H_%M_%S")
with open(f"benchmarking_results_{formatted_timestamp}.pkl", "wb") as res_file:
    pickle.dump(benchmarking.results, res_file)

Benchmarking:   0%|          | 0/6 [00:00<?, ?it/s]

the dataset is : compas-recidivism, the backend is : sklearn, the method is gaussian


100%|██████████| 1/1 [00:15<00:00, 15.71s/it]


the dataset is : compas-recidivism, the backend is : sklearn, the method is random


100%|██████████| 1/1 [00:59<00:00, 59.52s/it]


...

Diverse Counterfactuals found! total time taken: 03 min 20 sec
the dataset is : adult-income, the backend is : PYT, the method is spherical


Benchmarking:  83%|████████▎ | 5/6 [1:11:23<16:46, 1006.95s/it, dataset=adult-income, backend=PYT, method=spherical]    

Diverse Counterfactuals found! total time taken: 01 min 09 sec
the dataset is : adult-income, the backend is : TF2, the method is gaussian
Diverse Counterfactuals found! total time taken: 02 min 05 sec
the dataset is : adult-income, the backend is : TF2, the method is random
Diverse Counterfactuals found! total time taken: 04 min 25 sec
the dataset is : adult-income, the backend is : TF2, the method is spherical


Benchmarking: 100%|██████████| 6/6 [1:22:20<00:00, 823.34s/it, dataset=adult-income, backend=TF2, method=spherical] 

Diverse Counterfactuals found! total time taken: 04 min 17 sec


In [15]:
from dice_ml_x.utils import helpers
a = helpers.load_lending_club_dataset()
a.describe(include='all').transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sex,4966,2,Male,4008,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,4966.0,NaN,NaN,NaN,34.335884,11.667358,18.0,25.0,31.0,42.0,80.0
juv_misd_count,4966.0,NaN,NaN,NaN,0.103504,0.531543,0.0,0.0,0.0,0.0,13.0
priors_count,4966.0,NaN,NaN,NaN,3.552356,4.950223,0.0,0.0,2.0,5.0,38.0
twoyearrecid,4966.0,NaN,NaN,NaN,0.5,0.50005,0.0,0.0,0.5,1.0,1.0
age_cat,4966,3,25-45,2857,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race,4966,2,African-American,3004,NaN,NaN,NaN,NaN,NaN,NaN,NaN
c_charge_degree,4966,2,F,3248,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
from 

{'compas-recidivism': {'sklearn': {'accuracy': 0.6197183098591549,
   'cfs': {'gaussian':       sex   age  juv_misd_count  priors_count        age_cat  \
    0  Female  19.0             0.0          10.0  Greaterthan45   
    0    Male  19.0             0.0          10.0  Greaterthan45   
    0  Female  50.0             0.0          10.0  Greaterthan45   
    0  Female  45.0             0.0          10.0  Greaterthan45   
    
                   race c_charge_degree  twoyearrecid  
    0  African-American               F             0  
    0         Caucasian               F             0  
    0         Caucasian               F             0  
    0         Caucasian               F             0  ,
    'random':       sex   age  juv_misd_count  priors_count        age_cat  \
    0  Female  53.0             0.0          10.0  Greaterthan45   
    0    Male  49.0             0.0          11.0  Greaterthan45   
    0  Female  19.0             0.0           0.0  Greaterthan45   
    0 

In [9]:
d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=["age", "juv_misd_count", "priors_count"], outcome_name='twoyearrecid')
numerical = ["age", "juv_misd_count", "priors_count"]
categorical = x_train.columns.difference(numerical)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', RandomForestClassifier())])
model = clf.fit(x_train, y_train)

In [13]:
from sklearn.metrics import accuracy_score

predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 61.97%


In [38]:
# Using sklearn backend
m = dice_ml_x.Model(model=model, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml_x.DiceX(d, m, method="genetic")

In [39]:
cat_features = {}
for col in categorical:
    if col in df.columns:
        cat_features[col] = df[col].unique().tolist()
cat_features

{'age_cat': ['25-45', 'Lessthan25', 'Greaterthan45'],
 'c_charge_degree': ['F', 'M'],
 'race': ['African-American', 'Caucasian'],
 'sex': ['Male', 'Female']}

In [40]:
gaussian_kwargs = {
    'continuous_features': numerical,
    'categorical_features': cat_features,
    'std_dev': 0.3
}

random_kwargs = {
    'continuous_features': numerical,
    'categorical_features': cat_features,
    'feature_ranges': exp.data_interface.get_features_range_float()[1]
}

spherical_kwargs = {
    'continuous_features': numerical,
    'categorical_features': cat_features,
    'feature_ranges': exp.data_interface.get_features_range_float()[1]
}

e1 = exp.generate_counterfactuals(x_test[0:1], total_CFs=5, perturbation_method="gaussian", desired_class="opposite", **gaussian_kwargs)
e1.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [10:51<00:00, 651.37s/it]

Query instance (original outcome : 1)


,sex,age,juv_misd_count,priors_count,age_cat,race,c_charge_degree,twoyearrecid
0,Male,38.0,5.0,8.0,25-45,African-American,M,1



Diverse Counterfactual set (new outcome: 0)


,sex,age,juv_misd_count,priors_count,age_cat,race,c_charge_degree,twoyearrecid
0,-,40.0,0.0,-,Greaterthan45,-,-,0


In [41]:
import matplotlib.pyplot as plt

def plot_loss_metrics(loss_metrics):
    """Plots loss metrics over iterations or samples."""
    # Extract metrics
    proximity_loss = loss_metrics["proximity_loss"]
    sparsity_loss = loss_metrics["sparsity_loss"]
    robustness_loss = loss_metrics["robustness_loss"]
    yloss = loss_metrics["y_loss"]
    total_loss = loss_metrics["total_loss"]

    # Create a figure
    plt.figure(figsize=(10, 6))

    # Plot proximity loss
    plt.plot(proximity_loss, label="Proximity Loss", marker='o')

    # Plot sparsity loss
    plt.plot(sparsity_loss, label="Sparsity Loss", marker='s')

    # Plot robustness loss
    plt.plot(robustness_loss, label="Robustness Loss", marker='x')

    # Plot yloss
    plt.plot(yloss, label="Y Loss", marker='^')

    plt.plot(total_loss, label="Total Loss", marker='v')

    # Customize plot
    plt.title("Loss Metrics")
    plt.xlabel("Index (Iterations)")
    plt.ylabel("Loss Value")
    plt.legend()
    plt.grid()

    # Show plot
    plt.show()

plot_loss_metrics(exp.loss_history)

AttributeError: 'DiceGenetic' object has no attribute 'loss_history'